In [1]:
from dataclasses import dataclass

In [ ]:
@dataclass
class Person():
    name: str
    age:int
    city:str

In [6]:
person=Person(name="Sanjukta", age=5, city="Columbia")
print(person)

Person(name='Sanjukta', age=5, city='Columbia')


In [7]:
person=Person(name="Sanjukta", age=5, city=5)
print(person)

Person(name='Sanjukta', age=5, city=5)


This is the problem. Even if we pass an integer for city value, we are getting the response. No TYPECHECK is happening. Thus PYDANTIC is used

In [8]:
from pydantic import BaseModel

In [9]:
class Person1(BaseModel):
    name: str
    age: int
    city: str

In [ ]:
person=Person1(name="Sanjukta", age=5, city="Columbia")
print(person)

name='Sanjukta' age=5 city='Columbia'


In [11]:
person=Person1(name="Sanjukta", age=5, city=5)
print(person)

ValidationError: 1 validation error for Person1
city
  Input should be a valid string [type=string_type, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

# 2. Model with Optional Fields

Add optional fields using Python's Optional type:

In [12]:
from typing import Optional

In [13]:
class Employee(BaseModel):
    id:int
    name:str
    department:str
    salary:Optional[float]=None #Optional with default value
    is_active:Optional[bool]=None #Optional with default value

In [14]:
emp1=Employee(id=1, name="Sanjukta", department="CS")
print(emp1)

id=1 name='Sanjukta' department='CS' salary=None is_active=None


In [15]:
emp2=Employee(id=1, name="Sanjukta", department="CS", salary="900000")
print(emp2)

id=1 name='Sanjukta' department='CS' salary=900000.0 is_active=None


str is converted to floating values

Definitions:
- Optional[type]:Indicates the field can be None
- Default value (=None or = True):Makes the field optional
- Required fields must still bbe provided
- Pydantic validates types even for optional fields when values are provided

In [18]:
emp3=Employee(id=1, name="Sanjukta", department="CS", salary="900000", is_active=23)
print(emp3)

ValidationError: 1 validation error for Employee
is_active
  Input should be a valid boolean, unable to interpret input [type=bool_parsing, input_value=23, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/bool_parsing

In [19]:
from typing import List

In [20]:
class Classroom(BaseModel):
    room_number:str
    student: List[str] #List of strings
    capacity: int
    

In [24]:
classroom = Classroom(  
    room_number="A101",
    student=("Alice", "Bob", "Charlie"),
    capacity=30
)
print(classroom)

room_number='A101' student=['Alice', 'Bob', 'Charlie'] capacity=30


#### tuple --> list by pydantic

In [26]:
classroom = Classroom(  
    room_number="A101",
    student=("Alice", 50 , "Charlie"),
    capacity=30
)
print(classroom)

ValidationError: 1 validation error for Classroom
student.1
  Input should be a valid string [type=string_type, input_value=50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [ ]:
try:
    invalid_val= Classroom(  
    room_number="A101",
    student=("Alice", 50 , "Charlie"),
    capacity=30
    )
except ValueError as e:
    print(e)

1 validation error for Classroom
student.1
  Input should be a valid string [type=string_type, input_value=50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


## 3. Model with Nested Models

#### Create complex structires with nested models:

In [28]:
class Address(BaseModel):
    street:str
    city:str
    zip_code:str

class Customer(BaseModel):
    customer_id:int
    name:str
    address:Address ## Nested Model

In [30]:
customer=Customer(customer_id=1, name="Sanjukta", address={"street":"Heath Street", "city": "Columbia", "zip_code":"65203"})
print(customer)

customer_id=1 name='Sanjukta' address=Address(street='Heath Street', city='Columbia', zip_code='65203')


In [32]:
customer=Customer(customer_id=1, name="Sanjukta", address={"street":"Heath Street", "city": 123, "zip_code":"65203"})
print(customer)

ValidationError: 1 validation error for Customer
address.city
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [33]:
from pydantic import BaseModel, Field

In [34]:
class Item(BaseModel):
    name:str=Field(min_length=2, max_length=50)
    price:float=Field(gt=0, le=10000)
    quantity:int=Field(ge=0)

item=Item(name="Book", price=1000000, quantity=10)

ValidationError: 1 validation error for Item
price
  Input should be less than or equal to 10000 [type=less_than_equal, input_value=1000000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal

In [36]:
item=Item(name="Book", price=5000, quantity= 10)
print(item)

name='Book' price=5000.0 quantity=10


In [37]:
class User(BaseModel):
    username:str=Field(description="Unique username for the user")
    age:int=Field(default=18, description="User age default to 18")
    email:str=Field(default_factory=lambda: "user@example.com", description="Default email address")

#Examples
user1 = User(username="alice")
print(user1)

username='alice' age=18 email='user@example.com'


In [38]:
user2 = User(username="bob", age=25, email="bob@domain.com")
print(user2)

username='bob' age=25 email='bob@domain.com'


In [39]:
User.model_json_schema()

{'properties': {'username': {'description': 'Unique username for the user',
   'title': 'Username',
   'type': 'string'},
  'age': {'default': 18,
   'description': 'User age default to 18',
   'title': 'Age',
   'type': 'integer'},
  'email': {'description': 'Default email address',
   'title': 'Email',
   'type': 'string'}},
 'required': ['username'],
 'title': 'User',
 'type': 'object'}

In [ ]:
b 